# Классификация ЭКГ. Вариант 7
## Часть 1. Подготовка данных

Цель работы — подготовить выборку PTB-XL для распознавания нормы, желудочковой экстрасистолии и фибрилляции предсердий. В этой части выполнены загрузка данных в хранилище, анализ разметки, разбиение по пациентам и экспорт сигналов для обучения.

### Постановка задачи

| Код | Состояние |
|---|---|
| NORM | Нормальная ЭКГ |
| PVC | Желудочковая экстрасистолия |
| AFIB | Фибрилляция предсердий |

Метки рассматриваются в порядке `NORM`, `PVC`, `AFIB`. Они могут встречаться одновременно. Поэтому задача является многометочной, а не классификацией по трём взаимоисключающим классам. Запись без этих кодов имеет метку `[0, 0, 0]`.

In [1]:
import json
import os
import zipfile
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd
import pyarrow.csv as pacsv
from IPython.display import display

from ecg_warehouse.api import WarehouseAPI
from ecg_warehouse.dataset import DatasetSpec, SplitSpec, load_dataset, read_dataset_signals

if Path.cwd().name == "notebooks":
    os.chdir("..")

assert Path("src/ecg_warehouse").exists(), "Откройте ноутбук из проекта ecg-warehouse"
pd.options.display.float_format = "{:.4f}".format

### Параметры

Исходные файлы находятся в `data/ptb-xl`. Подготовленный датасет сохраняется в соседний проект `ecg-ml`, в каталог `data/ptb-xl-variant-7`. Пути ниже заданы относительно корня `ecg-warehouse`.

In [2]:
VARIANT = 7
CLASSES = ["NORM", "PVC", "AFIB"]
PTB_XL_ARCHIVE = Path("data/ptb-xl.zip")
PTB_XL_PATH = Path("data/ptb-xl")
DATASET_NAME = f"ptb-xl-variant-{VARIANT}"
DATASET_EXPORT_PATH = Path("../ecg-ml/data") / DATASET_NAME
REEXPORT = False  # True — заново записать уже существующий экспорт.

## 1. Исходные данные

Использован [PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/): 21 799 записей от 18 869 пациентов. Каждая ЭКГ содержит 12 отведений длительностью 10 секунд. В работе выбрана частота 500 Гц, то есть 5000 отсчётов на отведение. Если данные уже распакованы, повторная распаковка пропускается.

In [3]:
if not PTB_XL_PATH.exists():
    with zipfile.ZipFile(PTB_XL_ARCHIVE) as archive:
        members = [name for name in archive.namelist() if not name.startswith("__MACOSX/")]
        archive.extractall(PTB_XL_PATH.parent, members=members)

for name in ["ptbxl_database.csv", "scp_statements.csv", "records500"]:
    assert (PTB_XL_PATH / name).exists(), f"Не найден файл или каталог: {name}"

## 2. Загрузка в хранилище

Метаданные и диагнозы сохраняются в таблицах Apache Iceberg, сигналы — в Icechunk. Для работы с хранилищем используется `WarehouseAPI`. Локальные файлы хранилища размещены в `data/`; путь к таблицам задан абсолютным, чтобы он корректно обрабатывался в Windows.

In [4]:
local_config = json.loads(Path("cfg/local.json").read_text(encoding="utf-8"))
warehouse_path = Path("data").resolve()
local_config["warehouse"] = (
    warehouse_path.as_posix() if os.name == "nt" else warehouse_path.as_uri()
)
wh = WarehouseAPI(local_config)

### Загрузка записей

Загружаются ЭКГ PTB-XL с частотой 500 Гц и их диагнозы SCP-ECG. Если все записи уже присутствуют в хранилище, используются загруженные данные.

In [5]:
loaded = wh.sql("""
    SELECT count(*) AS recordings
    FROM recordings
    WHERE source = 'ptb-xl-1.0.3' AND sampling_rate_hz = 500
""").iloc[0, 0]

if loaded == 21799:
    print("В хранилище уже есть все 21 799 ЭКГ PTB-XL.")
else:
    wh.ingest_ptb_xl(data_dir=PTB_XL_PATH, rate=500)

В хранилище уже есть все 21 799 ЭКГ PTB-XL.


### Проверка объёма данных

SQL-запросом проверяются число записей, число пациентов, частота дискретизации и количество отведений.

In [6]:
display(
    wh.sql("""
    SELECT source, sampling_rate_hz, num_leads,
           count(*) AS recordings, count(DISTINCT patient_id) AS patients
    FROM recordings
    GROUP BY source, sampling_rate_hz, num_leads
""")
)

,source,sampling_rate_hz,num_leads,recordings,patients
0,ptb-xl-1.0.3,500,12,21799,18869


### Распределение меток

Для каждого кода подсчитывается число различных записей. Дополнительно рассматриваются сочетания меток: простое суммирование размеров классов не даёт число ЭКГ, поскольку классы пересекаются.

In [7]:
class_counts = (
    wh.sql("""
    SELECT code, any_value(label_text) AS label_text,
           count(DISTINCT recording_id) AS recordings
    FROM diagnoses
    WHERE code_system = 'SCP-ECG' AND code IN ('NORM', 'PVC', 'AFIB')
    GROUP BY code
""")
    .set_index("code")
    .loc[CLASSES]
    .reset_index()
)
total_recordings = int(wh.sql("SELECT count(*) AS n FROM recordings").iloc[0, 0])
class_counts["fraction"] = class_counts["recordings"] / total_recordings
display(class_counts)
ratio = class_counts["recordings"].max() / class_counts["recordings"].min()
print(f"Отношение размеров наибольшего и наименьшего класса: {ratio:.2f}")
combinations = wh.sql("""
    WITH labels AS (
        SELECT r.recording_id,
               max(CASE WHEN d.code = 'NORM' THEN 1 ELSE 0 END) AS NORM,
               max(CASE WHEN d.code = 'PVC' THEN 1 ELSE 0 END) AS PVC,
               max(CASE WHEN d.code = 'AFIB' THEN 1 ELSE 0 END) AS AFIB
        FROM recordings r LEFT JOIN diagnoses d ON r.recording_id = d.recording_id
        GROUP BY r.recording_id
    )
    SELECT NORM, PVC, AFIB, count(*) AS recordings FROM labels
    GROUP BY NORM, PVC, AFIB ORDER BY recordings DESC
""")
display(combinations)

,code,label_text,recordings,fraction
0,NORM,normal ECG,9514,0.4364
1,PVC,ventricular premature complex,1143,0.0524
2,AFIB,atrial fibrillation,1514,0.0695


Отношение размеров наибольшего и наименьшего класса: 8.32


,NORM,PVC,AFIB,recordings
0,0,0,0,10057
1,1,0,0,9290
2,0,0,1,1275
3,0,1,0,751
4,0,1,1,202
5,1,1,0,187
6,1,0,1,34
7,1,1,1,3


### Баланс классов

Положительных примеров NORM в 8,32 раза больше, чем PVC. У 10 057 ЭКГ нет ни одной из трёх меток; эти записи оставлены как отрицательные примеры. Сочетание только NORM и PVC встречается в 187 записях, только PVC и AFIB — в 202, только NORM и AFIB — в 34. Все три кода есть у трёх записей.

Основной метрикой выбрана macro-F1: в ней каждый класс имеет одинаковый вес. Во второй части также сравниваются обычная и взвешенная BCE; веса рассчитываются только по train.

## 3. Формирование выборки

В спецификации зафиксированы источник PTB-XL 1.0.3, 12 отведений, частота 500 Гц и длительность 10 секунд. Из диагнозов оставлены коды варианта 7. Записи с другими диагнозами не удаляются: для выбранных меток они могут быть отрицательными примерами.

Разбиение на train и test выполняется по пациентам с `seed=0`. Один пациент не должен попасть в обе части. Доля test задана равной 0,2; фактическое отношение числа записей может отличаться из-за разного количества ЭКГ у пациентов.

In [8]:
dataset_spec = DatasetSpec(
    name=DATASET_NAME,
    sources=["ptb-xl-1.0.3"],
    num_leads=12,
    sampling_rate_hz=500,
    code_systems=["SCP-ECG"],
    codes=CLASSES,
    min_duration_sec=10,
    max_duration_sec=10,
    recording_columns=[
        "recording_id",
        "source",
        "patient_id",
        "sampling_rate_hz",
        "duration_sec",
        "age_at_recording",
        "sex",
    ],
    diagnosis_columns=[
        "diagnosis_id",
        "recording_id",
        "code",
        "code_system",
        "label_text",
    ],
    split=SplitSpec(test_fraction=0.2, seed=0),
)

### Сборка датасета

По спецификации создаётся датасет `ptb-xl-variant-7`. Манифест содержит версию, параметры выборки и ссылки на файлы. Сигналы на этом этапе остаются в хранилище.

In [9]:
dataset = wh.build_dataset(spec=dataset_spec)
print(f"Датасет: {dataset['name']}@{dataset['version']}")
display(pd.DataFrame(dataset["artifacts"]["recordings"]["splits"]).T)

Built dataset ptb-xl-variant-7@0.0.0-dev.he06f7f2cf90902be (dev): 21799 recordings, 12171 diagnoses
Датасет: ptb-xl-variant-7@0.0.0-dev.he06f7f2cf90902be


,recordings,patients
train,17380,15084
test,4419,3785


## 4. Экспорт и проверка

Экспорт содержит `signals.npz` с сигналами, метками, порядком классов, идентификаторами и разбиением, а также `meta.csv` и `diagnoses.csv`.

Массив сигналов занимает 4,87 ГиБ. Поэтому сначала записываю его по одной ЭКГ во временный файл через `open_memmap`, затем собираю NPZ и удаляю временный файл. На время экспорта нужны дополнительные 4,87 ГиБ на диске. Чтение из хранилища идёт окнами по 16 записей с проверкой контрольных сумм.

Готовый экспорт используется повторно. Для пересоздания нужно задать `REEXPORT = True`. Ниже проверяю его идентификаторы и метки по текущему датасету, а также размеры сигналов и разбиение по пациентам.

In [10]:
def export_npz(manifest, recordings, diagnoses, output):
    output.mkdir(parents=True, exist_ok=True)
    recording_ids = recordings["recording_id"].to_pylist()
    row_of = {recording_id: i for i, recording_id in enumerate(recording_ids)}
    class_of = {name: i for i, name in enumerate(CLASSES)}
    labels = np.zeros((len(recording_ids), len(CLASSES)), dtype=np.uint8)
    for recording_id, code in zip(
        diagnoses["recording_id"].to_pylist(), diagnoses["code"].to_pylist()
    ):
        labels[row_of[recording_id], class_of[code]] = 1

    temporary = output / "X.partial.npy"
    signals = np.lib.format.open_memmap(
        temporary, mode="w+", dtype=np.float32, shape=(len(recording_ids), 12, 5000)
    )
    try:
        count = 0
        with patch("ecg_warehouse.recordings_store.READ_WINDOW", 16):
            for i, signal in enumerate(read_dataset_signals(manifest, recordings)):
                if signal.shape != (12, 5000) or not np.isfinite(signal).all():
                    raise ValueError(f"Некорректный сигнал в строке {i}")
                signals[i] = signal
                count += 1
        assert count == len(recording_ids), "Число сигналов не совпадает с метаданными"
        signals.flush()

        with (output / "signals.partial.npz").open("wb") as stream:
            np.savez(
                stream,
                X=signals,
                y=labels,
                classes=np.array(CLASSES),
                recording_id=np.array(recording_ids),
                split=np.array(recordings["split"].to_pylist()),
            )
        pacsv.write_csv(recordings, output / "meta.csv")
        pacsv.write_csv(diagnoses, output / "diagnoses.csv")
        (output / "signals.partial.npz").replace(output / "signals.npz")
    finally:
        del signals  # Закрываем отображение файла перед удалением в Windows.
        temporary.unlink(missing_ok=True)


manifest, recordings, diagnoses = load_dataset(wh.warehouse, dataset["name"], dataset["version"])
export_files = ["signals.npz", "meta.csv", "diagnoses.csv"]
if REEXPORT or not all((DATASET_EXPORT_PATH / name).exists() for name in export_files):
    export_npz(manifest, recordings, diagnoses, DATASET_EXPORT_PATH)
    print("Экспорт сохранён.")
else:
    print("Использован готовый экспорт.")

display(
    pd.DataFrame(
        {
            "Файл": export_files,
            "Размер, МиБ": [
                (DATASET_EXPORT_PATH / name).stat().st_size / 2**20 for name in export_files
            ],
        }
    )
)

Использован готовый экспорт.


,Файл,"Размер, МиБ"
0,signals.npz,4992.8683
1,meta.csv,3.9257
2,diagnoses.csv,1.2863


In [11]:
with zipfile.ZipFile(DATASET_EXPORT_PATH / "signals.npz") as archive:
    with archive.open("X.npy") as stream:
        version = np.lib.format.read_magic(stream)
        readers = {
            (1, 0): np.lib.format.read_array_header_1_0,
            (2, 0): np.lib.format.read_array_header_2_0,
        }
        signal_shape, _, signal_dtype = readers[version](stream)

with np.load(DATASET_EXPORT_PATH / "signals.npz") as arrays:
    labels = arrays["y"]
    export_classes = arrays["classes"].tolist()
    recording_ids = arrays["recording_id"].astype(str)
    splits = arrays["split"].astype(str)

assert export_classes == CLASSES
assert signal_shape == (21799, 12, 5000)
assert signal_dtype == np.dtype("float32")
assert labels.shape == (len(recording_ids), len(CLASSES))
assert len(set(recording_ids)) == len(recording_ids)
assert set(splits) == {"train", "test"}
assert recording_ids.tolist() == recordings["recording_id"].to_pylist()
assert splits.tolist() == recordings["split"].to_pylist()

diagnosis_table = diagnoses.to_pandas()
expected_labels = pd.crosstab(diagnosis_table["recording_id"], diagnosis_table["code"])
expected_labels = expected_labels.reindex(index=recording_ids, columns=CLASSES, fill_value=0)
assert np.array_equal(labels, (expected_labels.to_numpy() > 0).astype(np.uint8))

meta = pd.read_csv(
    DATASET_EXPORT_PATH / "meta.csv",
    dtype={"recording_id": str, "source": str, "patient_id": str},
)
assert not meta["recording_id"].duplicated().any()
meta = meta.set_index("recording_id").loc[recording_ids].reset_index()
assert (meta["split"].to_numpy() == splits).all()
assert meta["patient_id"].notna().all()

groups = meta["source"] + ":" + meta["patient_id"]
assert not set(groups[splits == "train"]) & set(groups[splits == "test"])
summary = []
for name in ["train", "test"]:
    mask = splits == name
    summary.append(
        {
            "split": name,
            "recordings": int(mask.sum()),
            "patients": int(groups[mask].nunique()),
            **{code: int(count) for code, count in zip(CLASSES, labels[mask].sum(axis=0))},
        }
    )

display(pd.DataFrame(summary))
print("Размеры, метки и разбиение проверены. Общих пациентов между train и test нет.")
_ = (DATASET_EXPORT_PATH / "lab_summary.json").write_text(
    json.dumps(
        {"variant": VARIANT, "classes": CLASSES, "shape": list(signal_shape), "split": summary},
        indent=2,
    ),
    encoding="utf-8",
)

,split,recordings,patients,NORM,PVC,AFIB
0,train,17380,15084,7603,919,1179
1,test,4419,3785,1911,224,335


Размеры, метки и разбиение проверены. Общих пациентов между train и test нет.


## Вывод

Подготовлены 21 799 ЭКГ с метками `NORM`, `PVC`, `AFIB`. После разбиения получены 17 380 записей train и 4419 записей test. Проверены размерность сигналов `(21799, 12, 5000)`, порядок меток и отсутствие общих пациентов между частями. Датасет экспортирован для обучения классификатора во второй части работы.